In [ ]:
import os
from torch.utils.data import Dataset, DataLoader

DATA_FOLDER = "data"
SICK_EN_FOLDER = "sick_en"
SICK_EN_FILE = "SICK_annotated.txt"

In [10]:
class SICKDataset(Dataset):
    def __init__(self, filepath, split):
        self.sentence_pairs = {}
        self.labels = {}

        self.load_sick_dataset(filepath, split)


    def load_sick_dataset(self, filepath, split):        
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"File not found: {filepath}")
        
        with open(filepath, "r", encoding="utf-8") as f:
            next(f) # Skip first line, since it is the column names
            for line in f:
                line = line.strip()
                if not line:  # Skip empty lines
                    continue
                    
                data = [s.strip() for s in line.split("\t")]
                
                if data[-1].lower() != split:
                    continue
                
                pair_ID = int(data[0])
                # pair_type = data[1]
                sentence_A = data[2]
                # sentence_A_expRule = data[3]
                sentence_B = data[4]
                # sentence_B_expRule = data[5]
                # relatedness_score = float(data[6])
                label = data[7].lower()
                # entailment_AB = data[8]
                # entailment_BA = data[9]
                # sentence_A_original = data[10]
                # sentence_B_original = data[11]
                # sentence_A_dataset = data[12]
                # sentence_B_datase = data[13]
                label = data[7].lower()
                
                self.sentence_pairs[pair_ID] = (sentence_A, sentence_B)
                self.labels[pair_ID] = label
            
    def __getitem__(self, index):
        return self.sentence_pairs[index], self.labels[index]



In [ ]:
sick_en_filepath = f"./{DATA_FOLDER}/{SICK_EN_FOLDER}/{SICK_EN_FILE}"

sick_en_dataset_train = SICKDataset(sick_en_filepath, "train")
sick_en_dataset_test = SICKDataset(sick_en_filepath, "test")
sick_en_dataset_val = SICKDataset(sick_en_filepath, "val")

sick_en_dataset_train[2]

(('A group of children is playing in the house and there is no man standing in the background',
  'A group of kids is playing in a yard and an old man is standing in the background'),
 'neutral')

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_FOLDER = "models"

olmo = AutoModelForCausalLM.from_pretrained("allenai/Olmo-3-1025-7B")
tokenizer = AutoTokenizer.from_pretrained("allenai/Olmo-3-1025-7B")

olmo_filepath = f"./{MODEL_FOLDER}/olmo_model"

olmo.save_pretrained(olmo_filepath)
tokenizer.save_pretrained(olmo_filepath)

config.json: 0.00B [00:00, ?B/s]

c:\Users\nicol\PycharmProjects\bachelor_thesis\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nicol\.cache\huggingface\hub\models--allenai--Olmo-3-1025-7B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow fie

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./models/olmo_model\\tokenizer_config.json',
 './models/olmo_model\\tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(olmo_filepath)
model = AutoModelForCausalLM.from_pretrained(olmo_filepath).to(device)

prompt = "The future of AI is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

The future of AI is here, and it's not just for tech experts anymore. With the rise of AI tools like ChatGPT, anyone can now harness the power of artificial intelligence to enhance their productivity and creativity. Whether you're a student, a professional, or just
